# Tutorial: Analysis of CASSCF Solutions

---

This tutorial shows how to interpret the results of a CASSCF calculation using different types of orbitals.
This tutorial assumes that you are already familiar with CASSCF claculations in Forte2.

### 💻 Importing the relevant modules from Forte2

To begin we will import a few components from Forte2:

In [ ]:
from pathlib import Path
import numpy as np

HAVE_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError as e:
    print(f"You likely need to install matplotlib: see original error message: {e}")
    HAVE_MPL = False

from forte2 import MCOptimizer, RHF, CISolver, State, System, write_orbital_cubes
from forte2.base_classes.params import DavidsonLiuParams

## 📚 Invariance of the CASSCF wavefunction with respect to orbital rotations

The CASSCF wavefunction is the sum of all the determinants in the active space ($| \Phi_I \rangle$) weighted by their corresponding CI coefficients ($c_I$):

\begin{equation}
| \Psi_\text{CASSCF} \rangle = \sum_I c_I | \Phi_I \rangle.
\end{equation}

In this expression, the Slater determinants are antisymmetrized products of spinorbitals ($\phi_i$), which we write as:

\begin{equation}
| \Phi_I \rangle = | \phi_{i_1} \phi_{i_2} \cdots \phi_{i_N} \rangle
\end{equation}

An important property of the CASSCF wavefunction is that it can be expressed in different orbital bases related by unitary transformations.
If we define a new set of spinorbitals ($\phi'_i$) as a linear combination of the original spinorbitals ($\phi_i$):

\begin{equation}
| \phi'_i \rangle = \sum_j U_{ij} | \phi_j \rangle,
\end{equation}

then we can defined a new set of Slater determinants ($| \Phi'_I \rangle$) in terms of the new spinorbitals:

\begin{equation}
| \Phi'_I \rangle = | \phi'_{i_1} \phi'_{i_2} \cdots \phi'_{i_N} \rangle.
\end{equation}

The CASSCF wavefunction can be expressed in terms of the new Slater determinants as:

\begin{equation}
| \Psi_\text{CASSCF} \rangle = \sum_I c'_I | \Phi'_I \rangle,
\end{equation}

where the new CI coefficients ($c'_I$) are related to the original ones by the unitary transformation.

This property of the CASSCF wavefunction implies that the orbitals are not uniquely defined.
**In order for an analysis of the CASSCF wavefunction to be well defined and reproducible**, we need to choose a specific orbital basis to represent the wavefunction.
Different options are available, and this tutorial will explore those implemented in Forte2.

## 📚 Semicanonical and natural orbitals

In this section we discuss two common methods for defining a consistent set of CASSCF orbitals:
1. **Semicanonical orbitals** are obtained by diagonalizing the active block of the generalized Fock matrix:
\begin{equation}
f_{pq} = h_{pq} + \sum_{rs} \langle pr||qs \rangle \gamma_{rs},
\end{equation}
where $h_{pq}$ is the one-electron Hamiltonian, $\langle pr||qs \rangle$ are the antisymmetrized two-electron integrals, and $\gamma_{rs}$ is the one-particle density matrix of the CASSCF wavefunction:
\begin{equation}
\gamma_{pq} = \langle \Psi_\text{CASSCF} | a^\dagger_p a_q | \Psi_\text{CASSCF} \rangle.
\end{equation}
Starting from general CASSCF orbitals, one forms the active block of the generalized Fock matrix and solves the eigenvalue problem:
\begin{equation}
\mathbf{f}^{\mathbb{A}} \mathbf{U}^{\mathbb{A}} = \mathbf{U}^{\mathbb{A}} \mathbf{\epsilon}^{\mathbb{A}}.
\end{equation}
The resulting eigenvectors $\mathbf{U}_{\mathbb{A}}$ determine the orbital transformation to obtain the semicanonical orbitals, while the corresponding eigenvalues $\mathbf{\epsilon}_{\mathbb{A}}$ are the **semicanonical orbital energies**.

2. **Natural orbitals**, which are obtained by diagonalizing the one-body density matrix of the CASSCF wavefunction in the active space:
\begin{equation}
\boldsymbol{\gamma}_1^{\mathbb{A}} \mathbf{U}^{\mathbb{A}} = \mathbf{U}^{\mathbb{A}} \mathbf{n}^{\mathbb{A}}.
\end{equation}
The eigenvalues $\mathbf{n}^{\mathbb{A}}$ are the **natural occupation numbers**, which are the eigenvalues of the one-particle density matrix in the active space.

In Forte2, **the default setting for CASSCF is to use semicanonical orbitals**. The code additionally uses semicanonical core and virtual orbitals, which are obtained by diagonalizing the core and virtual blocks of the generalized Fock matrix.

## 💻 Example: CO full-valence CAS(10,8)

To demonstrate the difference between semicanonical and natural orbitals, we run a full-valence CASSCF calculation on CO [CAS(10,8)].
The C and O 1s-like orbitals are kept doubly occupied.

### Semicanonical orbitals (default for CASSCF)

The following code block sets up a CASSCF calculation using the default value of the `final_orbitals` parameter, which is set to `semicanonical`. 

In [ ]:
xyz = f"""
C  0.0  0.0  0.0
O  0.0  0.0  2.5
"""

system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    cholesky_tei=True,
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

singlet = State(system=system, multiplicity=1, ms=0.0)

cas_solver = CISolver(
    states=singlet,
    core_orbitals=2,
    active_orbitals=8,
)

casscf_default = MCOptimizer(cas_solver, e_tol=1.0e-12)(rhf)

casscf_default.run()

When expressed in the semicanonical orbital basis, the CASSCF wavefunction contains many leading determinants with significant contributions to the wavefunction.
This can be seen in the following table printed out in the output, which shows the top determinants in the CASSCF wavefunction along with their CI coefficients.
```md
Top determinants:
===========================================================================
Contrib.  #1           #2           #3           #4           #5           
---------------------------------------------------------------------------
Root 0    |222b20a0>   |222a20b0>   |2222ba00>   |2222ab00>   |222aabb0>   
          -0.287714    -0.287714    +0.287705    +0.287705    +0.233169    
===========================================================================
```

### Details: Helper functions

A helper function is defined in the next code block and used in the rest of the notebook to plot the active block of the Fock matrix and the one-particle density matrix in the active space.

In [ ]:
def plot_fock_and_rdm(mc):
    if not HAVE_MPL:
        print("matplotlib is not available, skipping plotting")
        return

    # compute the 1-RDM
    g1 = mc.make_rdm(0,order=1, spin_type="sf")

    # compute the Fock matrix in the MO basis
    fock_builder = mc.system.fock_builder
    mo_space = mc.mo_space
    C_contig = mc.mos.C[0][:, mo_space.orig_to_contig]
    C_docc, C_act = C_contig[:, mo_space.docc], C_contig[:, mo_space.actv]
    fock_ao = fock_builder.build_generalized_fock(
        C_core=C_docc,
        C_act=C_act,
        g1=g1,
    )
    f_mo = C_contig.conj().T @ fock_ao @ C_contig    
    f_act = f_mo[mo_space.actv][:, mo_space.actv]
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    fock_range = np.max(np.abs(f_act.real))
    im0 = axes[0].imshow(f_act.real, cmap='seismic', vmin=-fock_range, vmax=fock_range)
    axes[0].set_title('Fock Matrix in MO Basis')
    axes[0].set_xlabel('Active orbitals')
    axes[0].set_ylabel('Active orbitals')
    axes[0].set_xticks(range(f_act.shape[0]))
    axes[0].set_yticks(range(f_act.shape[0]))
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(g1, cmap='seismic', vmin=-2.0, vmax=2.0)
    axes[1].set_title('1-RDM in MO Basis')
    axes[1].set_xlabel('Active orbitals')
    axes[1].set_ylabel('Active orbitals')
    axes[1].set_xticks(range(g1.shape[0]))
    axes[1].set_yticks(range(g1.shape[0]))
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    fig.tight_layout()
    plt.show()

To verify that the final orbitals are semicanonical, we can evaluate the active block of the Fock matrix. The function below will use the information contained in a MCOptimizer object to evaluate the Fock matrix in the final CASSCF orbitals and return the active block of the Fock matrix. The function will also compute the one-particle reduced density matrix (1-RDM) in the final CASSCF orbitals basis. These two quantities are then plotted for comparison.

In [ ]:
plot_fock_and_rdm(casscf_default)

### Natural orbitals

Next, we examine the natural orbitals.
In the following input, we change the default value of the `final_orbitals` argument to `"natural"` and rerun the CASSCF calculation. The resulting 1-RDM is then plotted along with the active block of the Fock matrix.

In [ ]:
system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    cholesky_tei=True,
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

singlet = State(system=system, multiplicity=1, ms=0.0)

cas_solver = CISolver(
    states=singlet,
    core_orbitals=2,
    active_orbitals=8,
    davidson_liu_params=DavidsonLiuParams(ndets_per_guess=100, maxiter=100)
)
casscf_nos = MCOptimizer(
    cas_solver,
    final_orbitals="natural",
)(rhf)
casscf_nos.run()

plot_fock_and_rdm(casscf_nos)

```md
Top determinants:
===========================================================================
Contrib.  #1           #2           #3           #4           #5           
---------------------------------------------------------------------------
Root 0    |22222000>   |2222ba00>   |2222ab00>   |222b20a0>   |222a20b0>   
          -0.511926    -0.223301    -0.223301    -0.223278    -0.223278    
===========================================================================
```

## Atomic-like localized orbitals

Forte2 provides two localized final-orbital representations. `"ibo"` localizes the active orbitals within each GAS. `"ibo_atomic"` additionally aligns atom-local IBO sets with projected, axis-oriented MINAO functions.

For `"ibo_atomic"`, the population of orbital $i$ on atom $A$ is
\begin{equation}
p_{Ai}=\sum_{\mu\in A}|\langle\mathrm{IAO}_\mu|\mathrm{IBO}_i\rangle|^2.
\end{equation}
IBOs whose dominant atomic population exceeds $\tau=0.9$ form a candidate set. The smallest eigenvalue of its atomic population matrix must also exceed $\tau$; this makes the locality test invariant to rotations within the set.

IAO populations determine atom locality. Column-pivoted QR selects independent projected MINAO targets, and an orthogonal Procrustes rotation maximizes their overlap with the IBOs. Alignment is skipped only when those targets are rank deficient. A weak individual target population produces a confidence warning, not a localization failure.

Both representations semicanonicalize inactive orbitals and order the final active orbitals by generalized-Fock energy. The following code runs CASSCF with `final_orbitals="ibo_atomic"`.

In [ ]:
system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    cholesky_tei=True,
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

singlet = State(system=system, multiplicity=1, ms=0.0)

cas_solver = CISolver(
    states=singlet,
    core_orbitals=2,
    active_orbitals=8,
    davidson_liu_params=DavidsonLiuParams(ndets_per_guess=100, maxiter=100)
)
casscf_ibo_atomic = MCOptimizer(
    cas_solver,
    final_orbitals="ibo_atomic",
)(rhf)
casscf_ibo_atomic.run()

plot_fock_and_rdm(casscf_ibo_atomic)

The IBO atomic-alignment summary shows the dominant atomic character of the final active orbitals.
These orbitals are ordered by their energy (from the CASSCF generalized Fock matrix).
```md
IBO atomic-alignment summary for GAS 1:
==========================================================================
  MO      Energy [Eh]   Atomic target   Target pop.   Main IAO   Main pop.
--------------------------------------------------------------------------
   3      -1.24129976   O1 2s                0.9964   O1 2s         0.9996
   4      -0.70564864   C1 2s                0.9976   C1 2s         0.9998
   5      -0.36482698   O1 2pz               0.9942   O1 2pz        1.0000
   6      -0.35520193   O1 2py               0.9996   O1 2py        1.0000
   7      -0.35520164   O1 2px               0.9996   O1 2px        1.0000
   8      -0.06563308   C1 2px               0.9990   C1 2px        1.0000
   9      -0.06563287   C1 2py               0.9990   C1 2py        1.0000
  10      -0.05127287   C1 2pz               0.9922   C1 2pz        1.0000
==========================================================================
```
We can see that these orbitals are localized on the C and O atoms, and are dominated by C1(2/3s), C1(2py), C1(2pz), C1(2px), O1(2/3s), O1(2py), O1(2pz), and O1(2px) atomic orbitals.

In this basis, the CASSCF wavefunction can be expressed in terms of determinants where the occupied orbitals have well defined atomic character. For example, the table below show that the largest contributions to the CASSCF wavefunction is a configuration of the form:
\begin{equation}
| (2s^\mathrm{O})^2 (2s^\mathrm{C})^2 (2p_z^\mathrm{O})^2 
 (2p_{y}^\mathrm{O})^1  (2p_x^\mathrm{O})^1  (2p_{y}^\mathrm{C})^1 (2p_x^\mathrm{C})^1 \rangle
\end{equation}

```md
Top determinants:
===========================================================================
Contrib.  #1           #2           #3           #4           #5           
---------------------------------------------------------------------------
Root 0    |222aabb0>   |222bbaa0>   |22a2ab0b>   |22b2ba0a>   |22bb20aa>   
          +0.324454    +0.324454    +0.304968    +0.304968    -0.304967      
===========================================================================
```

## Cube files for all orbitals

The following code block generates cube files for all orbitals in the calculation.

In [ ]:
for label, type in [("semicanonical", casscf_default), ("natural", casscf_nos), ("ibo_atomic", casscf_ibo_atomic)]:
    print(f"Writing orbital cubes for {label} orbitals...")
    write_orbital_cubes(
        system=system,
        C=type.mos.C[0],
        indices=list(range(10)),
        filepath=Path(f"co_cubes/{label}"),
    )